In [1]:
import findspark
findspark.init()
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LocalNotebookApp1") \
    .getOrCreate()




25/08/04 17:11:54 WARN Utils: Your hostname, testnode1 resolves to a loopback address: 127.0.1.1; using 192.168.1.144 instead (on interface eth0)
25/08/04 17:11:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/04 17:11:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# 2. Leer datos desde CSV
path = "/var/snp-dwh/sftp/sftp-data/cantidad_habitantes.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

df.printSchema()
df


root
 |-- _c0: integer (nullable = true)
 |-- nombre_provincia: string (nullable = true)
 |-- nombre_canton: string (nullable = true)
 |-- codigo_provincia: string (nullable = true)
 |-- codigo_canton: string (nullable = true)
 |-- año: integer (nullable = true)
 |-- cantidad_habitantes: integer (nullable = true)
 |-- mes_num: integer (nullable = true)
 |-- mes: string (nullable = true)
 |-- miles_habitantes: double (nullable = true)



DataFrame[_c0: int, nombre_provincia: string, nombre_canton: string, codigo_provincia: string, codigo_canton: string, año: int, cantidad_habitantes: int, mes_num: int, mes: string, miles_habitantes: double]

In [3]:
df.count()

13260

In [4]:
# ============================
# 3. Crear UDFs para lógica personalizada
#    Ejemplo: Clasificar provincias según cantidad de habitantes
# ============================

from pyspark.sql.functions import udf

def clasificar_poblacion(habitantes):
    if habitantes is None:
        return "Desconocido"
    elif habitantes < 50000:
        return "Pequeña"
    elif habitantes < 200000:
        return "Mediana"
    else:
        return "Grande"
clasificar_udf = udf(clasificar_poblacion)

# Agregar columna con categoría de población
df = df.withColumn("categoria_poblacion", clasificar_udf(F.col("cantidad_habitantes")))

In [5]:
# ============================
# 4. Filtrar y transformar el DataFrame
#    - Filtrar solo registros del año 2024 y mes > 6
#    - Calcular una nueva columna 'habitantes_k' (habitantes en miles)
# ============================

df_filtrado = df.filter((F.col("año") == 2024) & (F.col("mes_num") > 6)) \
    .withColumn("habitantes_k", (F.col("cantidad_habitantes") / 1000).cast("double"))

df_filtrado.show(5)


+---+----------------+-------------+----------------+-------------+----+-------------------+-------+----------+----------------+-------------------+------------+
|_c0|nombre_provincia|nombre_canton|codigo_provincia|codigo_canton| año|cantidad_habitantes|mes_num|       mes|miles_habitantes|categoria_poblacion|habitantes_k|
+---+----------------+-------------+----------------+-------------+----+-------------------+-------+----------+----------------+-------------------+------------+
| 43|           AZUAY|       CUENCA|               1|          101|2024|             619701|      7|     Julio|         619.701|             Grande|     619.701|
| 44|           AZUAY|       CUENCA|               1|          101|2024|             619701|      8|    Agosto|         619.701|             Grande|     619.701|
| 45|           AZUAY|       CUENCA|               1|          101|2024|             619701|      9|Septiembre|         619.701|             Grande|     619.701|
| 46|           AZUAY|      

25/08/04 17:11:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , nombre_provincia, nombre_canton, codigo_provincia, codigo_canton, año, cantidad_habitantes, mes_num, mes, miles_habitantes
 Schema: _c0, nombre_provincia, nombre_canton, codigo_provincia, codigo_canton, año, cantidad_habitantes, mes_num, mes, miles_habitantes
Expected: _c0 but found: 
CSV file: file:///var/snp-dwh/sftp/sftp-data/cantidad_habitantes.csv


In [6]:
# ============================
# 5. Agrupar y agregar resultados
#    - Calcular la suma total y promedio de habitantes por provincia
# ============================

df_agg = df_filtrado.groupBy("nombre_provincia") \
    .agg(
        F.sum("cantidad_habitantes").alias("total_habitantes"),
        F.avg("cantidad_habitantes").alias("promedio_habitantes")
    ) \
    .orderBy(F.desc("total_habitantes"))

df_agg.show()


+----------------+----------------+-------------------+
|nombre_provincia|total_habitantes|promedio_habitantes|
+----------------+----------------+-------------------+
|          GUAYAS|        28438626|          189590.84|
|       PICHINCHA|        19633590|         409033.125|
|          MANABI|        10196604|            77247.0|
|        LOS RIOS|         5811960|  74512.30769230769|
|           AZUAY|         4987320| 55414.666666666664|
|          EL ORO|         4491762| 53473.357142857145|
|      ESMERALDAS|         3609756|  85946.57142857143|
|      TUNGURAHUA|         3474492| 64342.444444444445|
|   SANTO DOMINGO|         3141144|           261762.0|
|            LOJA|         2984628|          31089.875|
|        IMBABURA|         2964210|  82339.16666666667|
|        COTOPAXI|         2934882|  69878.14285714286|
|      CHIMBORAZO|         2933142|            48885.7|
|     SANTA ELENA|         2420868| 134492.66666666666|
|           CAÑAR|         1424820|  33924.28571

In [7]:
# ============================
# 6. Guardar en formato Parquet
# ============================

output_path = "/var/snp-dwh/sftp/output/habitantes_procesado.parquet"

df_agg.write.mode("overwrite").parquet(output_path)

print(f"✅ Datos agregados y guardados en formato Parquet en: {output_path}")

# Finalizar sesión Spark
#spark.stop()

✅ Datos agregados y guardados en formato Parquet en: /var/snp-dwh/sftp/output/habitantes_procesado.parquet


1. Leer el Parquet guardado

In [8]:
# 1. Volver a leer el Parquet ya procesado
df_parquet = spark.read.parquet("/var/snp-dwh/sftp/output/habitantes_procesado.parquet")

print("✅ DataFrame leído desde Parquet:")
df_parquet.show(5)

✅ DataFrame leído desde Parquet:
+----------------+----------------+-------------------+
|nombre_provincia|total_habitantes|promedio_habitantes|
+----------------+----------------+-------------------+
|          GUAYAS|        28438626|          189590.84|
|       PICHINCHA|        19633590|         409033.125|
|          MANABI|        10196604|            77247.0|
|        LOS RIOS|         5811960|  74512.30769230769|
|           AZUAY|         4987320| 55414.666666666664|
+----------------+----------------+-------------------+
only showing top 5 rows

